# FNO Darcy Fourier-mode study on a Kaggle GPU

Launcher for [physicsnemo-fno-darcy](https://github.com/AdebanjiAdelowo/physicsnemo-fno-darcy). It checks out one commit of the repository and calls its scripts; it contains no model or training code.

Session settings: **Accelerator** an NVIDIA GPU, **Internet** on.

| Step | What runs | When |
|---|---|---|
| A | generator checks on CUDA (`scripts/verify_generator.py --device cuda`) | always |
| | tests, CUDA smoke study, one-pseudo-epoch timing probe | always |
| B | main study, `configs/full.yaml` (256 × 256, `IndependentDarcy2D`) | `RUN_FULL = True` |
| C | upstream-behaviour control, `configs/official.yaml` (`Darcy2D` as shipped) | `RUN_OFFICIAL = True` |

Before B and before C the notebook projects the runtime from a probe and compares it with what is left of `SESSION_BUDGET_HOURS`. If the projection does not fit, the step is **not started and nothing is reduced**; the notebook reports the numbers and finishes.

In [ ]:
REPO_URL = "https://github.com/AdebanjiAdelowo/physicsnemo-fno-darcy.git"
REF = "SET_TO_FULL_40_CHARACTER_COMMIT_SHA"   # output of `git rev-parse HEAD` for the commit to run
RUN_FULL = False                              # step B: only the value True starts it
RUN_OFFICIAL = False                          # step C: only the value True starts it
SESSION_BUDGET_HOURS = 9.0                    # GPU time you are willing to spend in this session; check Kaggle's current limits
GITHUB_TOKEN_SECRET = None                    # name of a Kaggle secret holding a GitHub token; only for a private repository

import re, time
T0 = time.time()
assert re.fullmatch(r"[0-9a-f]{40}", REF), "REF must be a full 40-character commit SHA, not a branch or tag."
WORK = "/kaggle/working"
REPO = f"{WORK}/physicsnemo-fno-darcy"

def hours_left():
    return SESSION_BUDGET_HOURS - (time.time() - T0) / 3600

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "CUDA is not available. Select a GPU accelerator; this notebook does not fall back to CPU."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
import os, subprocess
def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

if not os.path.isdir(REPO):
    url = REPO_URL
    if GITHUB_TOKEN_SECRET:
        from kaggle_secrets import UserSecretsClient
        url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET) + "@")
    subprocess.run(["git", "clone", "--quiet", url, REPO], check=True)   # not echoed: the URL may hold a token
    subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--quiet", REF], cwd=REPO, check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True).stdout.strip()
assert head == REF, f"checked out {head}, expected {REF}"
assert not dirty, "the working tree is not clean"
print("commit", head)

The requirements pin PhysicsNeMo to a commit of its main branch. `pip` may replace the PyTorch build of the image if it is older than PhysicsNeMo requires; the next cell prints the versions afterwards, in a new process, and checks CUDA again.

In [ ]:
sh("pip install -q -r requirements.txt", cwd=REPO)
sh('python -c "import torch, physicsnemo, warp; assert torch.cuda.is_available(); '
   "print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| physicsnemo', physicsnemo.__version__, '| warp', warp.__version__, '|', torch.cuda.get_device_name(0))\"", cwd=REPO)

## Step A: generator checks on CUDA

Runs before any training. It measures, with the Warp CUDA backend, (1) how many distinct permeability fields a batch of `Darcy2D` holds, as shipped and with `IndependentDarcy2D`, and (2) the distance of the generated pressure from sparse direct solves of the kernel stencil and of the conservative Darcy equation. The GPU and the CUDA, Warp, PyTorch and PhysicsNeMo versions are stored in the JSON file.

In [ ]:
import json, zipfile
CHECK = f"{WORK}/generator-check-cuda"   # outside the clone, so the tree stays clean for the studies
sh(f"python scripts/verify_generator.py --device cuda --out-root {CHECK}", cwd=REPO)
check = json.load(open(f"{CHECK}/results/generator_check_cuda.json"))
env = check["environment"]
print({k: env[k] for k in ("gpu_name", "cuda_version", "torch_version", "warp_version", "physicsnemo")})
shipped = check["diversity"]["Darcy2D"]
repeated = max(shipped["distinct_fields_per_batch"]) == 1
print(f"Darcy2D as shipped on CUDA: distinct fields per batch of {shipped['batch_size']}: {shipped['distinct_fields_per_batch']}, "
      f"distinct coefficient values: {shipped['distinct_coefficient_values_per_batch']} of {shipped['coefficients_per_batch']}")
print("REPEATED-BATCH DEFECT ON CUDA:", "REPRODUCED" if repeated else "NOT REPRODUCED (compare with results/generator_check.json, CPU)")
with zipfile.ZipFile(f"{WORK}/physicsnemo-fno-darcy-generator-check-cuda.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for rel in ("results/generator_check_cuda.json", "figures/generator_check_cuda.png"):
        z.write(f"{CHECK}/{rel}", rel)
print("written:", f"{WORK}/physicsnemo-fno-darcy-generator-check-cuda.zip")
assert not subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True).stdout.strip(), "step A modified the clone"

In [ ]:
sh("python -m pytest -q", cwd=REPO)

## CUDA smoke study

Two tiny models, two pseudo-epochs. The numbers are not results.

In [ ]:
sh("python scripts/train.py --config smoke device=cuda name=smoke_cuda", cwd=REPO)
sh("python scripts/plot_results.py --study runs/smoke_cuda --out runs/smoke_cuda/figures", cwd=REPO)

## Timing probe for step B

Generates the datasets of the full study (they are cached and reused in step B) and trains the largest model for one pseudo-epoch. The projection is an extrapolation from that one pseudo-epoch, with every run costed as the 16-mode model.

In [ ]:
sh("python scripts/train.py --config full name=full_probe study.fno_modes=[16] study.seeds=[0] "
   "training.max_pseudo_epochs=1 training.rec_results_freq=1 validation.validation_pseudo_epochs=1", cwd=REPO)
probe = json.load(open(f"{REPO}/runs/full_probe/modes16_seed0/train_metrics.json"))
meta = json.load(open(f"{REPO}/runs/full_probe/study_metadata.json"))
n_runs = 12  # 4 mode counts x 3 seeds in configs/full.yaml
full_projection_h = probe["train_seconds"] * 60 * n_runs / 3600  # 60 pseudo-epochs in configs/full.yaml
print(f"one pseudo-epoch, 16 modes: {probe['train_seconds']:.1f} s on {meta['gpu_name']}")
print("data generation: " + ", ".join(f"{k} {v['generation_seconds']:.0f} s" for k, v in meta["datasets"].items()))
print(f"PROJECTION for step B: {full_projection_h:.1f} h. Left of the session budget: {hours_left():.1f} h.")

## Step B: main study (`configs/full.yaml`)

In [ ]:
full_done = False
if RUN_FULL is not True:
    print("Step B skipped: RUN_FULL is False.")
elif full_projection_h > hours_left():
    print(f"STEP B NOT STARTED: projected {full_projection_h:.1f} h, {hours_left():.1f} h left of SESSION_BUDGET_HOURS = {SESSION_BUDGET_HOURS}.")
    print("The configuration was not reduced. Decide how to proceed before changing configs/full.yaml.")
else:
    sh("python scripts/train.py --config full --resume", cwd=REPO)
    sh("python scripts/plot_results.py --study runs/full --out runs/full/figures", cwd=REPO)
    sh("python scripts/results.py package runs/full", cwd=REPO)
    sh("python scripts/results.py bundle runs/full", cwd=REPO)
    sh(f"cp runs/full.zip {WORK}/physicsnemo-fno-darcy-full.zip && cp runs/full_eval_bundle.zip {WORK}/physicsnemo-fno-darcy-full-eval-bundle.zip", cwd=REPO)
    full_done = True
    print("STEP B COMPLETE.")

## Step C: upstream-behaviour control (`configs/official.yaml`)

The NVIDIA example's settings: 12 modes, 256 pseudo-epochs of 2048 newly generated samples, `Darcy2D` as shipped, default solver tolerance. It runs only after step B has completed in this session.

In [ ]:
official_done = False
if RUN_OFFICIAL is not True:
    print("Step C skipped: RUN_OFFICIAL is False.")
elif not full_done:
    print("STEP C NOT STARTED: step B did not complete in this session, and it has priority.")
else:
    sh("python scripts/train.py --config official name=official_probe "
       "training.max_pseudo_epochs=1 training.rec_results_freq=1 validation.validation_pseudo_epochs=1", cwd=REPO)
    probe = json.load(open(f"{REPO}/runs/official_probe/modes12_seed0/train_metrics.json"))
    official_projection_h = probe["train_seconds"] * 256 / 3600  # 256 pseudo-epochs in configs/official.yaml
    print(f"PROJECTION for step C: {official_projection_h:.1f} h. Left of the session budget: {hours_left():.1f} h.")
    if official_projection_h > hours_left():
        print("STEP C NOT STARTED: it does not fit in the remaining budget. The configuration was not reduced.")
    else:
        sh("python scripts/train.py --config official --resume", cwd=REPO)
        sh("python scripts/plot_results.py --study runs/official --out runs/official/figures", cwd=REPO)
        sh("python scripts/results.py package runs/official", cwd=REPO)
        sh("python scripts/results.py bundle runs/official", cwd=REPO)
        sh(f"cp runs/official.zip {WORK}/physicsnemo-fno-darcy-official.zip && cp runs/official_eval_bundle.zip {WORK}/physicsnemo-fno-darcy-official-eval-bundle.zip", cwd=REPO)
        official_done = True
        print("STEP C COMPLETE.")

In [ ]:
import glob
print("Archives in /kaggle/working:")
for path in sorted(glob.glob(f"{WORK}/physicsnemo-fno-darcy-*.zip")):
    print(f"  {path}  {os.path.getsize(path) / 2**20:.1f} MB")
print(f"Elapsed: {(time.time() - T0) / 3600:.2f} h. Step A: done. Step B: {'done' if full_done else 'not run'}. Step C: {'done' if official_done else 'not run'}.")
if not full_done:
    print("KAGGLE CUDA CHECKS COMPLETE. No full study was run.")